# Lesson 12 · Simulating replications: when does empirical Bayes fail?

**Source:** *Introduction to Empirical Bayes: Examples from Baseball Statistics*, David Robinson (2017), Chapter 13 (pp. 130–141)

**Question:** One simulation could be a lucky draw. Does empirical Bayes work *consistently*, and how many observations do you need before you can trust it?

**Goal:** Repeat Lesson 11's simulation 50 times, then vary the number of players from 30 to 10,000 and see where the method breaks down, and why.

**Contents**

1. Setup
2. 50 replications
3. Are the hyperparameters estimated well?
4. Accuracy, coverage and FDR across replications
5. Varying the number of players
6. Why small samples go wrong
7. Exercises

**Tags:** `simulation` `replication` `sample-size` `calibration` `hyperparameter-uncertainty` `robustness`

**Before you start:** Lesson 11. This lesson runs a few hundred fits, so add progress printing to your loops; each fit should take well under a second.

*How this notebook works:* each **Your turn** prompt is followed by an empty cell for your code. **Check** boxes give values to compare against; they come from the book, and your numbers can differ slightly because the Lahman data now runs through more recent seasons. **Reflect** prompts are for short written answers. Every prompt is numbered *lesson.section.question* (for example **2.3.1**), so you can refer to it; empty code cells and answer cells carry the same number.

In [ ]:
# --- Setup: run this cell first ------------------------------------------
# Windows + conda: put the environment's DLL folders on PATH, so plotting
# can't crash the kernel when Jupyter wasn't started from the activated env.
# (If ENVIRONMENT_NOTES.md in the Cardprice folder has a PATH-fix block,
# you can paste that here instead.)
import os, sys
for sub in ["Library\\mingw-w64\\bin", "Library\\usr\\bin", "Library\\bin", "Scripts", "bin"]:
    p = os.path.join(sys.prefix, sub)
    if os.path.isdir(p) and p not in os.environ["PATH"]:
        os.environ["PATH"] = p + os.pathsep + os.environ["PATH"]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy import stats, optimize, special

import eb_data   # helpers in this folder: loading the Lahman data, saving results

rng = np.random.default_rng(2017)
pd.set_option("display.precision", 4)
plt.rcParams["figure.figsize"] = (7, 4)
print(f"numpy {np.__version__} | pandas {pd.__version__} | scipy {scipy.__version__}")

%load_ext autoreload
%autoreload 2
import time
try:
    import my_eb   # your toolkit from Lesson 10
except ImportError:
    print("my_eb.py not found: finish Lesson 10, or define the functions you need below.")

career = eb_data.career(pitcher_rule=3)[["playerID", "H", "AB"]]
print(f"{len(career):,} players")

## 1. Setup

**12.1.1 · Your turn.** Fit the prior to all players and store the true `alpha0`, `beta0` and mean for the simulations.

In [ ]:
# 12.1.1 · Your code here

## 2. 50 replications

A tidy way to replicate: stack 50 copies of the players with a `replication` column (`pd.concat` or a cross merge), simulate p and H for all rows at once, then `groupby("replication")` to fit each one.

**12.2.1 · Your turn.** Simulate 50 replications. Fit a prior in each, printing progress (for example every 10 replications, with the elapsed time). Collect `alpha`, `beta` and `mean` per replication.

In [ ]:
# 12.2.1 · Your code here

In [ ]:
# 12.2.1 · Your code here

## 3. Are the hyperparameters estimated well?

**12.3.1 · Your turn.** Plot the 50 estimates of α₀, β₀ and the mean (strip or box plots), with the true values as dashed lines.

In [ ]:
# 12.3.1 · Your code here

> **Check:** the estimated means cluster tightly around the true mean (≈ 0.251).

## 4. Accuracy, coverage and FDR across replications

**12.4.1 · Your turn.** For each replication, compute the MSE of raw and shrunken estimates, and plot the two distributions.

In [ ]:
# 12.4.1 · Your code here

**12.4.2 · Your turn.** For each replication, compute the coverage of 95% intervals (histogram), then coverage curves for levels 0.5–0.95: one faint line per replication, plus y = x.

In [ ]:
# 12.4.2 · Your code here

In [ ]:
# 12.4.2 · Your code here

> **Check:** 95% coverage stays roughly between 94.4% and 95.5% across replications.

**12.4.3 · Your turn.** For each replication, plot the true false discovery proportion against the q-value threshold, one faint line each.

In [ ]:
# 12.4.3 · Your code here

**12.4.4 · Reflect.** FDR control promises the *expected* proportion, not the proportion in every run. What do your FDR curves show?

*12.4.4 · Your answer:*

## 5. Varying the number of players

**12.5.1 · Your turn.** For sizes 30, 100, 300, 1,000, 3,000 and 10,000, with 50 replications each, sample that many players' AB (with replacement), simulate p and H, and fit a prior. Print progress per size, and store the fitted priors along with each replication's MSE and 95% coverage.

In [ ]:
# 12.5.1 · Your code here

In [ ]:
# 12.5.1 · Your code here

**12.5.2 · Your turn.** Plot the hyperparameter estimates, MSE and coverage by size (box plots with log-spaced sizes).

In [ ]:
# 12.5.2 · Your code here

In [ ]:
# 12.5.2 · Your code here

**12.5.3 · Reflect.** Based on your plots, what's the smallest number of observations you'd be comfortable running empirical Bayes on? Why does the *spread* of the MSE matter more than its average when you only get one real dataset?

*12.5.3 · Your answer:*

## 6. Why small samples go wrong

The prior's spread is controlled by $\alpha_0 + \beta_0$: the larger the sum, the narrower the prior. Overestimate it and you over-shrink, so the intervals come out too narrow.

**12.6.1 · Your turn.** Plot each replication's estimated α₀ + β₀ (log x) against its 95% coverage, one colour per size, with a vertical line at the true value. Find the worst replication and look at its data.

In [ ]:
# 12.6.1 · Your code here

**12.6.2 · Reflect.** What does this show about the main weakness of empirical Bayes, the plug-in prior? How would a full Bayesian model treat the same 30 players differently? (That's Lesson 13.)

*12.6.2 · Your answer:*

## 7. Exercises

**12.7.1 · Your turn.** Halve every player's AB before simulating. How do MSE and coverage change?

In [ ]:
# 12.7.1 · Your code here

**12.7.2 · Your turn.** **Card connection.** There are only about 100–200 Pokémon sets. Using these results, would you trust empirical Bayes for set-level pull rates? Simulate 120 sets with realistic packs-opened counts to find out.

In [ ]:
# 12.7.2 · Your code here

## References

- *Introduction to Empirical Bayes: Examples from Baseball Statistics*, David Robinson (2017), Chapter 13.
- Carlin & Louis, *Bayesian Methods for Data Analysis* (3rd ed.), Chapter 5: empirical Bayes and its corrections.